# MiniGPT4 部署模型的测试

(使用环境是minigptv)

In [7]:
import sys
sys.path.append('/home/NCUT/teacher/xmy/MRAG/PMI/hugging_cache/MiniGPT-4')

In [8]:
from types import SimpleNamespace
from minigpt4.common.config import Config
from minigpt4.common.registry import registry
from minigpt4.conversation.conversation import Chat, CONV_VISION_Vicuna0, CONV_VISION_LLama2, StoppingCriteriaSub
from transformers import StoppingCriteriaList
import torch

def load_minigpt4_model(cfg_path='/home/NCUT/teacher/xmy/MRAG/PMI/hugging_cache/MiniGPT-4/eval_configs/minigpt4_eval.yaml', gpu_id=3):
    args = SimpleNamespace(
        cfg_path=cfg_path,
        options=[],
        gpu_id=gpu_id
    )
    cfg = Config(args)

    model_config = cfg.model_cfg
    model_config.device_8bit = args.gpu_id
    model_cls = registry.get_model_class(model_config.arch)
    model = model_cls.from_config(model_config).to(f'cuda:{gpu_id}')

    conv_dict = {
        'pretrain_vicuna0': CONV_VISION_Vicuna0,
        'pretrain_llama2': CONV_VISION_LLama2
    }
    CONV_VISION = conv_dict[model_config.model_type]

    vis_processor_cfg = cfg.datasets_cfg.cc_sbu_align.vis_processor.train
    vis_processor = registry.get_processor_class(vis_processor_cfg.name).from_config(vis_processor_cfg)

    stop_words_ids = [[835], [2277, 29937]]
    stop_words_ids = [torch.tensor(ids).to(f'cuda:{gpu_id}') for ids in stop_words_ids]
    stopping_criteria = StoppingCriteriaList([StoppingCriteriaSub(stops=stop_words_ids)])

    chat = Chat(model, vis_processor, device=f'cuda:{gpu_id}', stopping_criteria=stopping_criteria)

    return chat, CONV_VISION


In [9]:
from PIL import Image

def minigpt4_answer(chat, CONV_VISION, image: Image.Image, question: str) -> str:
    chat_state = CONV_VISION.copy()
    img_list = []
    chat.upload_img(image, chat_state, img_list)
    chat.encode_img(img_list)
    chat.ask(question, chat_state)

    answer = chat.answer(
        conv=chat_state,
        img_list=img_list,
        num_beams=1,
        temperature=1.0,
        max_new_tokens=300,
        max_length=2000
    )[0]

    return answer


In [10]:
from PIL import Image

# 模型初始化（只需一次）
chat, conv_template = load_minigpt4_model()




Loading checkpoint shards:   0%|          | 0/2 [00:08<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 44.00 MiB. GPU 3 has a total capacty of 44.40 GiB of which 42.19 MiB is free. Process 534029 has 21.56 GiB memory in use. Process 539136 has 10.03 GiB memory in use. Including non-PyTorch memory, this process has 12.74 GiB memory in use. Of the allocated memory 11.84 GiB is allocated by PyTorch, and 392.68 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
# 加载图片与提问
image = Image.open('/home/NCUT/teacher/xmy/MRAG/PMI/test/url2.jpg')
question = "describe this picture"

# 获取回答
answer = minigpt4_answer(chat, conv_template, image, question)
print("回答:", answer)

/home/NCUT/teacher/xmy/anaconda3/envs/minigptv/lib/python3.9/site-packages/transformers/generation/utils.py:1270: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation )
  warnings.warn(


回答: This is a picture of a large white house with a tall arched entrance and white shutters. The front lawn is covered in blooming flowers, and the trees on either side of the house are leafy and green. The roof has two large dormer windows, and there are several windows on the second floor. There are also two small balconies on the top floor. The house appears to be in good condition, and it is surrounded by lush greenery.


In [ ]:
# 加载图片与提问
image = Image.open('/home/NCUT/teacher/xmy/MRAG/PMI/test/url1.jpg')
question = "describe this picture"

# 获取回答
answer = minigpt4_answer(chat, conv_template, image, question)
print("回答:", answer)

回答: The Eiffel Tower is a tall, iron tower located in Paris, France. It was designed by Gustave Eiffel and completed in 1889 for the World's Fair. The tower stands at a height of 324 meters and has become one of the most iconic landmarks in the world, attracting millions of visitors every year.

The Eiffel Tower is known for its stunning views of the city, particularly during sunset when the clouds gather around it, casting dramatic shadows on the tower's surface. Despite its initial criticism, the tower has since become a beloved symbol of Paris and a must-see attraction for tourists visiting the city.

The Eiffel Tower's design features wrought iron beams and openwork balconies that provide visitors with breathtaking panoramic views of the city below. Visitors can climb up the tower to the top, where they can enjoy 360-degree views of Paris from an elevation of 279 meters.

In addition to its stunning architecture and views, the Eiffel Tower also holds significance in the history of 